# Problem1: Decision Frameworks: Housing & Furniture Sector
### Bayesian · Frequentist · Interventional Perspectives

**Dataset:** Daily closing prices of the **Housing sector ETF** (`house_series`) and the
**Furniture / Home-decor sector ETF** (`furniture_series`), covering
**April 2017 – February 2026** (2,239 trading days).

---

> *"The difference between an estimation problem and a decision problem is that in the latter,
> a specific action must be chosen from a defined set — and that choice has consequences."*
> — Wald (1950), *Statistical Decision Functions*

This notebook moves from passive parameter estimation to **active decision-making** under uncertainty.
The same dataset is examined through three philosophically distinct lenses, each prescribing
a different decision procedure and — crucially — potentially a different final action.


---
## 1 · Problem Formulation

### 1.1 The Decision Context

A portfolio manager at a multi-sector equity fund holds **furniture/home-decor sector
exposure** and uses the **housing sector as an economic signal**.
Every day, the manager observes the housing sector's return and must answer:

> **"Should I take a long position in the furniture sector ETF, or stay out of it?"**

This is a prototypical **binary statistical decision problem**.

### 1.2 Formal Setup

| Component | Definition |
|---|---|
| **Observable signal** | $x_t = r_t^{\text{house}}$, today's housing log-return |
| **Outcome of interest** | $y_t = r_t^{\text{furn}}$, today's furniture log-return |
| **Action space** | $\mathcal{A} = \{A_1: \text{Go Long Furniture},\ A_0: \text{Stay Out}\}$ |
| **Unknown state** | $\beta$ — the sensitivity of furniture returns to housing returns |
| **Structural model** | $y_t = \alpha + \beta x_t + \varepsilon_t$ |

The manager's problem: is $\beta$ large enough, and is the evidence strong enough,
to justify taking the long position?

### 1.3 Three Frameworks, Three Answers

Each framework interprets "the evidence is strong enough" differently:

| Framework | Central Question | Decision Criterion |
|---|---|---|
| **Frequentist** | Does $\beta \neq 0$ at a controlled error rate? | Reject $H_0$ at significance level $\alpha$ |
| **Bayesian** | What action minimises my expected loss given the posterior? | Minimise Bayes risk $\mathbb{E}[L(a, \beta) \mid \mathbf{y}]$ |
| **Interventional** | If I *force* housing returns to change, what happens? | Use causal $\beta$ (not observational) to size positions |

### 1.4 Why the Distinction Matters

The Frequentist and Bayesian frameworks are both **associational** — they describe
co-movement in observed data. The Interventional framework asks a fundamentally
different question: **if I act on the housing market** (e.g., a policy maker applies
stimulus, or a trader takes a housing ETF position intending to move it), what is the
*causal* knock-on effect on furniture?

This distinction matters because: observing $x_t$ and $y_t$ move together can
happen both because (a) housing *causes* furniture demand, and (b) a shared macro
factor (interest rates, consumer sentiment) drives *both simultaneously*.
The causal answer strips away (b), leaving only (a).


---
## 2 · Data Loading & Exploratory Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import HuberRegressor
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# ── Load & compute log-returns ─────────────────────────────────────────────────
df_furn  = pd.read_csv('furniture_series.csv')
df_house = pd.read_csv('house_series.csv')
for df in [df_furn, df_house]:
    df['Date'] = pd.to_datetime(df['Date'], utc=True)

merged = pd.merge(
    df_furn[['Date','Close']], df_house[['Date','Close']],
    on='Date', suffixes=('_furn','_house')
).sort_values('Date').reset_index(drop=True)

merged['ret_furn']  = np.log(merged['Close_furn']).diff()
merged['ret_house'] = np.log(merged['Close_house']).diff()
merged = merged.dropna().reset_index(drop=True)

y = merged['ret_furn'].values
x = merged['ret_house'].values
n = len(y)

print(f"Observations   : {n:,} trading days")
print(f"Date range     : {merged['Date'].iloc[0].date()} → {merged['Date'].iloc[-1].date()}")
print(f"Correlation    : {np.corrcoef(x,y)[0,1]:.4f}")
print(f"Kurtosis (furn): {stats.kurtosis(y, fisher=False):.2f}  (Gaussian = 3)")
print(f"Max drawdown   : {y.min()*100:.1f}% on {merged['Date'].iloc[y.argmin()].date()}")


In [ ]:
# ── EDA: price paths + return scatter + cross-return signal ──────────────────
fig = plt.figure(figsize=(15, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

# Price levels
ax0 = fig.add_subplot(gs[0, :2])
ax0b = ax0.twinx()
ax0.plot(merged['Date'], merged['Close_house'], '#1565C0', lw=0.9, label='Housing (left axis)')
ax0b.plot(merged['Date'], merged['Close_furn'],  '#E65100', lw=0.9, label='Furniture (right axis)')
covid = merged.loc[y.argmin(), 'Date']
ax0.axvline(covid, color='crimson', lw=1.2, ls='--', label='COVID crash')
ax0.set_title('Sector Price Levels', fontweight='bold')
ax0.set_ylabel('Housing Price ($)', color='#1565C0')
ax0b.set_ylabel('Furniture Price ($)', color='#E65100')
h1,l1 = ax0.get_legend_handles_labels(); h2,l2 = ax0b.get_legend_handles_labels()
ax0.legend(h1+h2, l1+l2, loc='upper left', fontsize=8)

# Returns time-series
ax1 = fig.add_subplot(gs[1, :2])
ax1.plot(merged['Date'], x, '#1565C0', lw=0.35, alpha=0.8, label='Housing')
ax1.plot(merged['Date'], y, '#E65100', lw=0.35, alpha=0.8, label='Furniture')
ax1.axvline(covid, color='crimson', lw=1.2, ls='--')
ax1.set_title('Daily Log-Returns', fontweight='bold')
ax1.set_ylabel('Log-return')
ax1.legend(fontsize=8)

# Scatter with OLS fit
ax2 = fig.add_subplot(gs[:, 2])
ax2.scatter(x, y, alpha=0.15, s=5, color='steelblue', label='Daily returns')
ax2.scatter(x[y.argmin()], y[y.argmin()], color='crimson', s=60, zorder=5, label='COVID crash')
xx = np.linspace(x.min(), x.max(), 200)
ols_quick = sm.OLS(y, sm.add_constant(x)).fit()
ax2.plot(xx, ols_quick.params[0] + ols_quick.params[1]*xx,
         'k-', lw=2, label=f'OLS β={ols_quick.params[1]:.3f}')
ax2.set_xlabel('Housing log-return')
ax2.set_ylabel('Furniture log-return')
ax2.set_title('Cross-Sector\nReturn Scatter', fontweight='bold')
ax2.legend(fontsize=8)

plt.suptitle('Exploratory Analysis: Housing × Furniture Sectors (2017–2026)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('eda_decision.png', bbox_inches='tight')
plt.show()
print("Key observation: strong positive co-movement, amplified during the COVID crash.")


---
## 3 · Frequentist Decision Framework

### 3.1 Conceptual Foundations

The Frequentist decision framework is rooted in **Neyman–Pearson hypothesis testing**.
Parameters are fixed (unknown) constants. The decision rule is designed to control
long-run error rates over hypothetical repeated sampling — not to quantify uncertainty
about the parameter itself.

**Two types of error and their costs:**

| | $H_0$ true ($\beta \leq 0$) | $H_1$ true ($\beta > 0$) |
|---|---|---|
| **Reject $H_0$** (Trade) | Type I Error (false alarm) — cost: transaction costs, strategy loss | Correct decision |
| **Fail to reject** (No Trade) | Correct decision | Type II Error (missed opportunity) — cost: foregone profit |

We fix $\alpha = 0.05$ (the maximum tolerable Type I error rate), then construct the most
powerful test at that level.

### 3.2 Hypothesis Test Specification

$$H_0: \beta \leq 0 \qquad \text{vs.} \qquad H_1: \beta > 0$$

This is a **one-sided test** — economically, we only care whether the relationship is
*positive* (justifying a *long* position). A negative $\beta$ would actually imply a
*short* trade, which is a different strategy and a different decision.

**Test statistic** (under $H_0$, with $\beta = 0$):
$$T = \frac{\hat{\beta}}{\widehat{\text{SE}}(\hat{\beta})} \sim t_{n-2}$$

**Decision rule:** Reject $H_0$ if $T > t_{\alpha,\, n-2}$

### 3.3 Power Analysis

Power = $P(\text{Reject } H_0 \mid H_1 \text{ is true}) = 1 - \beta_{\text{error}}$

$$\text{Power}(\beta^*) = P\left(T > t_{\alpha} \mid \beta = \beta^*\right) =
1 - \Phi\left(t_{\alpha} - \frac{\beta^* \sqrt{n}}{\sigma_\varepsilon / \sigma_x}\right)$$

where $\beta^*$ is the assumed true value under $H_1$.


In [ ]:
# ── OLS fit and one-sided t-test ───────────────────────────────────────────────
X_design = sm.add_constant(x)
ols_fit   = sm.OLS(y, X_design).fit()

beta_hat  = ols_fit.params[1]
se_beta   = ols_fit.bse[1]
t_stat    = beta_hat / se_beta
p_two     = ols_fit.pvalues[1]
p_one     = p_two / 2          # one-sided (t_stat >> 0, so p_one = p_two/2)
ci_95     = ols_fit.conf_int().iloc[1]
alpha_crit = stats.t.ppf(0.95, df=n-2)   # one-sided critical value at α=0.05

print("=" * 56)
print("  Frequentist Decision Test  (H0: β ≤ 0  vs  H1: β > 0)")
print("=" * 56)
print(f"  β̂              : {beta_hat:.4f}")
print(f"  SE(β̂)          : {se_beta:.4f}")
print(f"  t-statistic    : {t_stat:.2f}")
print(f"  Critical value : {alpha_crit:.2f}  (α=0.05, one-sided)")
print(f"  p-value (one)  : {p_one:.2e}")
print(f"  95% CI for β   : [{ci_95.iloc[0]:.4f}, {ci_95.iloc[1]:.4f}]")
print(f"  R²             : {ols_fit.rsquared:.4f}")
print("=" * 56)
print()
if t_stat > alpha_crit:
    print(f"  ✓ Reject H₀  →  DECISION: GO LONG FURNITURE")
    print(f"    (t={t_stat:.1f} >> critical value {alpha_crit:.2f})")
else:
    print("  ✗ Fail to reject H₀  →  DECISION: STAY OUT")


In [ ]:
# ── Power curve ───────────────────────────────────────────────────────────────
sigma_x   = np.std(x)
sigma_eps = np.std(ols_fit.resid)

beta_grid = np.linspace(0, 0.8, 300)
ncp       = beta_grid * sigma_x * np.sqrt(n) / sigma_eps   # non-centrality parameter
power_arr = stats.norm.sf(alpha_crit - ncp)

# ── Rolling 252-day t-statistic ────────────────────────────────────────────────
window = 252
roll_beta = []; roll_t = []; roll_dates = []
for i in range(window, n):
    xi, yi = x[i-window:i], y[i-window:i]
    Xi = sm.add_constant(xi)
    m  = sm.OLS(yi, Xi).fit()
    roll_beta.append(m.params[1])
    roll_t.append(m.tvalues[1])
    roll_dates.append(merged['Date'].iloc[i])
roll_beta  = np.array(roll_beta)
roll_t     = np.array(roll_t)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Frequentist Decision Framework', fontsize=13, fontweight='bold')

# Power curve
axes[0].plot(beta_grid, power_arr, '#1565C0', lw=2)
axes[0].axvline(beta_hat, color='crimson', ls='--', lw=1.5, label=f'Observed β={beta_hat:.3f}')
axes[0].axhline(0.80, color='grey', ls=':', lw=1, label='80% power')
axes[0].axhline(0.05, color='orange', ls=':', lw=1, label='α=0.05 (size)')
axes[0].set_xlabel('True β')
axes[0].set_ylabel('Power')
axes[0].set_title('Power Curve (n=2,238)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.05)

# Rolling beta
axes[1].plot(roll_dates, roll_beta, '#1565C0', lw=0.8)
axes[1].axhline(beta_hat, color='crimson', ls='--', lw=1.5, label=f'Full-sample β={beta_hat:.3f}')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].fill_between(roll_dates, roll_beta, 0,
                     where=(np.array(roll_beta)>0), alpha=0.15, color='#1565C0')
axes[1].set_title('Rolling 252-day β Estimate')
axes[1].set_ylabel('β')
axes[1].legend(fontsize=8)

# Rolling t-statistic
crit = stats.t.ppf(0.95, df=window-2)
axes[2].plot(roll_dates, roll_t, '#E65100', lw=0.8)
axes[2].axhline(crit, color='crimson', ls='--', lw=1.5, label=f'Critical value ({crit:.2f})')
axes[2].fill_between(roll_dates, roll_t, crit,
                     where=(np.array(roll_t) > crit), alpha=0.25, color='green',
                     label='Significant (trade signal)')
axes[2].set_title('Rolling t-statistic (one-sided)')
axes[2].set_ylabel('t')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('freq_decision.png', bbox_inches='tight')
plt.show()

pct_significant = (np.array(roll_t) > crit).mean()
print(f"Rolling windows with significant β: {pct_significant:.1%} of all 252-day periods")
print(f"Min rolling β: {roll_beta.min():.4f}  |  Max rolling β: {roll_beta.max():.4f}")
print(f"β was always positive: {(roll_beta > 0).all()}")


In [ ]:
# ── Trading decision rule backtest ─────────────────────────────────────────────
# Decision rule: "if r_house_t > 0, go long r_furn_t+1"
# (use today's housing return signal to decide tomorrow's furniture position)
# Evaluate over days 253 onwards (after initial estimation window)

signal   = (x > 0).astype(int)               # 1 = buy signal, 0 = no trade
pnl      = signal[:-1] * y[1:]               # P&L: signal today, return tomorrow
baseline = y[252+1:]                          # buy-and-hold furniture (benchmark)
pnl_eval = pnl[252:]                          # skip burn-in

# Hit rate and Sharpe
hit_rate = (pnl_eval > 0).mean()
sharpe_signal   = pnl_eval.mean() / pnl_eval.std() * np.sqrt(252)
sharpe_baseline = baseline.mean() / baseline.std() * np.sqrt(252)

print("=" * 50)
print("  Frequentist Trading Decision: Backtest")
print("=" * 50)
print(f"  Decision rule   : Long furniture if r_house > 0")
print(f"  Hit rate        : {hit_rate:.3f}  (50% = random)")
print(f"  Strategy Sharpe : {sharpe_signal:.3f}")
print(f"  B&H Sharpe      : {sharpe_baseline:.3f}")
print("=" * 50)
print()
print("Interpretation:")
print(f"  The frequentist test (p << 0.001) tells us β > 0 exists in the data.")
print(f"  A simple cross-sectional signal achieves a {hit_rate:.1%} directional hit rate,")
print(f"  slightly above random — consistent with a real but noisy relationship.")


---
## 4 · Bayesian Decision Framework

### 4.1 Conceptual Foundations

The Bayesian framework treats $\beta$ as a **random variable** with a prior distribution
$p(\beta)$ reflecting beliefs *before* seeing the data. After observing the data, we update
to the **posterior** $p(\beta \mid \mathbf{y})$ via Bayes' theorem:

$$p(\beta \mid \mathbf{y}) \propto p(\mathbf{y} \mid \beta) \cdot p(\beta)$$

But in the **decision** problem, the posterior is not the end — it is the *input* to a
decision-theoretic calculation. We must choose an action $a \in \mathcal{A}$ that
**minimises the posterior expected loss** (Bayes risk):

$$a^* = \arg\min_{a \in \mathcal{A}} \mathbb{E}_{\beta \mid \mathbf{y}}[L(a, \beta)]
      = \arg\min_{a} \int L(a, \beta)\, p(\beta \mid \mathbf{y})\, d\beta$$

This is the Bayesian decision rule — it is **optimal** in the sense of minimising
average loss with respect to the posterior distribution.

### 4.2 Prior Specification

We use a **weakly informative prior** that encodes our economic beliefs without
dominating the data:

$$\alpha \sim \mathcal{N}(0,\ 0.1^2) \qquad \text{(near-zero intercept expected)}$$
$$\beta  \sim \mathcal{N}(1,\ 0.5^2) \qquad \text{(positive co-movement expected; centered at 1)}$$
$$\sigma \sim \text{Half-Normal}(0.05) \qquad \text{(daily return scale)}$$

With $n = 2{,}238$ observations, the **Bernstein–von Mises theorem** guarantees the
posterior will be dominated by the likelihood, so the prior choice has minimal impact.

### 4.3 Asymmetric Loss Function

The key advantage of Bayesian decision theory is the ability to encode
**asymmetric consequences**. For our portfolio problem:

$$L(a, \beta) = \begin{cases}
c_{\text{FP}} & \text{if } a = A_1 \text{ (Trade) and } \beta \leq 0 \quad \text{(false positive: lose on trade)} \\
c_{\text{FN}} & \text{if } a = A_0 \text{ (No Trade) and } \beta > 0 \quad \text{(false negative: miss profit)} \\
0 & \text{otherwise}
\end{cases}$$

We consider three loss ratio scenarios: symmetric ($c_{\text{FP}} = c_{\text{FN}}$),
asymmetric-cautious ($c_{\text{FP}} \gg c_{\text{FN}}$, transaction-cost-heavy market),
and asymmetric-aggressive ($c_{\text{FP}} \ll c_{\text{FN}}$, high opportunity cost market).


In [ ]:
# ── Metropolis-Hastings MCMC ───────────────────────────────────────────────────
np.random.seed(42)

def log_posterior(params, y, x):
    alpha, beta, sigma = params
    if sigma <= 0: return -np.inf
    resid = y - alpha - beta * x
    ll = -n * np.log(sigma) - 0.5 * np.sum(resid**2) / sigma**2
    lp = stats.norm.logpdf(alpha, 0, 0.1)
    lp += stats.norm.logpdf(beta,  1, 0.5)   # prior centred at 1
    lp += stats.halfnorm.logpdf(sigma, scale=0.05)
    return ll + lp

n_iter  = 50_000
burnin  = 10_000
prop_sd = np.array([0.001, 0.010, 0.001])
chain   = np.zeros((n_iter, 3))
curr    = np.array([0.0, 0.6, 0.018])
accept  = 0

for i in range(n_iter):
    prop = curr + np.random.randn(3) * prop_sd
    lr   = log_posterior(prop, y, x) - log_posterior(curr, y, x)
    if np.log(np.random.rand()) < lr:
        curr = prop; accept += 1
    chain[i] = curr

post = chain[burnin:]
post_beta = post[:, 1]

print(f"Acceptance rate : {accept/n_iter:.2%}")
print(f"Posterior β     : mean={post_beta.mean():.4f}, sd={post_beta.std():.4f}")
print(f"95% Credible CI : [{np.percentile(post_beta,2.5):.4f}, {np.percentile(post_beta,97.5):.4f}]")
print(f"P(β > 0 | data) : {(post_beta > 0).mean():.6f}  (essentially 1)")
print(f"P(β > 0.1|data) : {(post_beta > 0.1).mean():.6f}  (essentially 1)")


In [ ]:
# ── Bayes risk computation under asymmetric loss ──────────────────────────────

# Three decision scenarios with different cost ratios
scenarios = {
    'Symmetric\n(c_FP = c_FN)':         (5, 5),
    'Cautious\n(c_FP = 10 × c_FN)':    (10, 1),
    'Aggressive\n(c_FP = 0.5 × c_FN)': (1, 2),
}

print("=" * 60)
print("  Bayesian Decision Under Asymmetric Loss")
print("=" * 60)
print(f"  {'Scenario':<28} {'E[L|Trade]':>11} {'E[L|No Trade]':>14} {'Decision':>10}")
print("-" * 60)

decisions = {}
for label, (c_fp, c_fn) in scenarios.items():
    # E[L | Trade]    = c_FP × P(β ≤ 0 | data)  — pay when strategy fails
    # E[L | No Trade] = c_FN × P(β > 0 | data)  — pay when opportunity missed
    E_trade    = c_fp * (post_beta <= 0).mean()
    E_no_trade = c_fn * (post_beta > 0).mean()
    action     = 'TRADE' if E_trade < E_no_trade else 'NO TRADE'
    decisions[label] = (E_trade, E_no_trade, action)
    name = label.replace('\n', ' ')
    print(f"  {name:<28} {E_trade:>11.6f} {E_no_trade:>14.6f} {action:>10}")
print("=" * 60)
print()
print("Since P(β > 0 | data) ≈ 1.0, the optimal action is TRADE")
print("under ALL three loss specifications.")


In [ ]:
# ── Sensitivity analysis: minimum c_FP/c_FN ratio to stay cautious ────────────
# Find the threshold cost ratio where decision flips to NO TRADE
# E[Trade] = c_FP * P(β<=0) < E[No Trade] = c_FN * P(β>0)
# ↔ c_FP / c_FN < P(β>0) / P(β<=0)  (posterior odds)
P_pos = (post_beta > 0).mean()
P_neg = 1 - P_pos
posterior_odds = P_pos / max(P_neg, 1e-10)
print(f"Posterior odds  P(β>0)/P(β≤0) = {posterior_odds:.0e}")
print(f"Decision flips to NO TRADE only if c_FP/c_FN > {posterior_odds:.0e}")
print()
print("In other words: you would need the cost of a false long to be ~10^236 times")
print("larger than the cost of missing the trade — economically impossible.")
print("The Bayesian decision is unambiguously: TRADE.")


In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Bayesian Decision Framework', fontsize=13, fontweight='bold')

# Posterior density with decision-relevant probabilities
ax = axes[0]
b_lo, b_hi = np.percentile(post_beta, [2.5, 97.5])
ax.hist(post_beta, bins=80, density=True, color='#1565C0', alpha=0.7)
ax.axvline(post_beta.mean(), color='crimson', lw=2, label=f'Posterior mean={post_beta.mean():.4f}')
ax.axvline(0, color='black', lw=1.5, ls='--', label='H₀ boundary (β=0)')
ax.axvspan(b_lo, b_hi, alpha=0.12, color='green', label=f'95% CI [{b_lo:.3f}, {b_hi:.3f}]')
ax.fill_between(
    np.linspace(-0.15, 0, 100),
    stats.norm.pdf(np.linspace(-0.15, 0, 100), post_beta.mean(), post_beta.std()),
    0, alpha=0.4, color='orange', label=f'P(β≤0) ≈ {P_neg:.1e}')
ax.set_xlabel('β')
ax.set_title('Posterior Distribution of β')
ax.legend(fontsize=7.5)
ax.set_xlim(-0.15, 0.85)

# Loss function surface
ax = axes[1]
c_fp_grid = np.logspace(-1, 2, 200)
E_trade_arr    = [(c * P_neg) for c in c_fp_grid]
E_no_trade_arr = [1.0 * P_pos for _ in c_fp_grid]   # fix c_FN=1
ax.plot(c_fp_grid, E_trade_arr,    '#1565C0', lw=2, label='E[Loss | Trade]')
ax.plot(c_fp_grid, E_no_trade_arr, '#E65100', lw=2, ls='--', label='E[Loss | No Trade]')
ax.set_xscale('log')
ax.set_xlabel('Cost of False Positive  c_FP  (log scale)')
ax.set_ylabel('Expected Loss (c_FN = 1)')
ax.set_title('Bayes Risk vs Loss Ratio
(Decision never flips in practice)')
ax.legend(fontsize=9)

# Prior vs Posterior comparison
ax = axes[2]
prior_samples = np.random.normal(1, 0.5, 100_000)
ax.hist(prior_samples, bins=100, density=True, alpha=0.4, color='orange', label='Prior β~N(1, 0.5²)')
ax.hist(post_beta,     bins=80,  density=True, alpha=0.7, color='#1565C0', label='Posterior')
ax.axvline(0, color='black', lw=1.2, ls='--')
ax.set_xlabel('β')
ax.set_title('Prior → Posterior Update
(Data overwhelms prior)')
ax.set_xlim(-2, 3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('bayes_decision.png', bbox_inches='tight')
plt.show()


---
## 5 · Interventional Decision Framework

### 5.1 Conceptual Foundations: The do-Operator

The Frequentist and Bayesian frameworks both answer an **associational** question:
*"Given that I observe housing returns equal $x$, what do I expect of furniture?"*
— i.e., $P(r_t^{\text{furn}} \mid r_t^{\text{house}} = x)$.

The Interventional framework, pioneered by Judea Pearl's **do-calculus**, asks a
fundamentally different question:

> *"What would happen to furniture returns if I **intervened** to set housing returns
> to $x$ — cutting housing off from its natural causes and forcing it to $x$ directly?"*

Formally: $P(r_t^{\text{furn}} \mid \mathbf{do}(r_t^{\text{house}} = x))$

The difference is not philosophical — it has direct practical consequences:

- A **portfolio manager** using the signal observationally is correct to use $\hat{\beta}_{\text{obs}} = 0.534$
- A **policy maker** who stimulates the housing sector (injecting only a *housing-specific* shock)
  should expect $\beta_{\text{causal}} < \beta_{\text{obs}}$, because the shared macro exposure
  that inflates the observational correlation is NOT activated by a sector-specific intervention

### 5.2 The Causal DAG (Directed Acyclic Graph)

```
       Macro Factor M_t
      (interest rates,
       consumer sentiment)
           /        \
          ↓          ↓
   r_house_t ──────→ r_furn_t
   (housing return)  (furniture return)
        ↑
   Housing-specific
   shock ε_house
```

- **M_t** is a **confounder** — it causes both $r_t^{\text{house}}$ and $r_t^{\text{furn}}$
- **$\varepsilon_{\text{house}}$** is the housing-*specific* (idiosyncratic) shock
- The **direct causal path** is $r_t^{\text{house}} \rightarrow r_t^{\text{furn}}$ (housing leads furniture demand)
- The **backdoor path** is $r_t^{\text{house}} \leftarrow M_t \rightarrow r_t^{\text{furn}}$ (spurious correlation)

Observational regression captures **both** paths; `do(r_house)` cuts the arrow from M to r_house,
leaving only the direct path.

### 5.3 Structural Causal Model (SCM)

$$M_t \sim \mathcal{N}(0,\ \sigma_M^2) \qquad \text{(unobserved macro factor)}$$
$$r_t^{\text{house}} = \gamma_h \cdot M_t + \varepsilon_t^{\text{house}}, \quad
  \varepsilon^{\text{house}} \perp M \qquad \text{(structural housing equation)}$$
$$r_t^{\text{furn}} = \underbrace{\beta_{\text{direct}}}_{\text{causal}} \cdot r_t^{\text{house}}
  + \gamma_f \cdot M_t + \varepsilon_t^{\text{furn}} \qquad \text{(structural furniture equation)}$$

The **observational** beta (what OLS estimates):
$$\beta_{\text{obs}} = \beta_{\text{direct}} + \underbrace{\frac{\gamma_h \cdot \gamma_f \cdot \sigma_M^2}{\text{Var}(r^{\text{house}})}}_{{\text{confounding bias}}}$$

The **interventional** beta (causal effect of `do(r_house)`):
$$\beta_{\text{causal}} = \beta_{\text{direct}}$$

### 5.4 Backdoor Adjustment Formula

If M is **observed**, we can recover $\beta_{\text{causal}}$ by:

$$P(r^{\text{furn}} \mid \mathbf{do}(r^{\text{house}} = x)) =
  \int P(r^{\text{furn}} \mid r^{\text{house}} = x,\ M = m)\, P(M = m)\, dm$$

In the linear Gaussian case, this reduces to: **regress $r^{\text{furn}}$ on $r^{\text{house}}$ and $M$
jointly**, and the coefficient on $r^{\text{house}}$ is $\beta_{\text{causal}}$.


In [ ]:
# ── Structural simulation calibrated to real data ─────────────────────────────
# We observe: Var(x)=0.000321, Var(y)=0.000320, Cov(x,y)=0.000229 → β_obs=0.534
# We POSIT (for illustration):
#   β_direct = 0.30  (true causal channel, housing demand → furniture demand)
#   The remainder of β_obs = 0.534 - 0.30 = 0.234 comes from confounding via M

np.random.seed(123)
N_sim       = 5_000          # simulated trading days
sigma_M     = 0.012          # macro factor volatility
gamma_h     = 0.85           # housing loading on macro
gamma_f     = 0.70           # furniture loading on macro
beta_direct = 0.30           # TRUE causal effect we want to recover

# Derived standard deviations (calibrated so Var(x_sim) ≈ Var(x_real))
sigma_eh = np.sqrt(max(np.var(x) - gamma_h**2 * sigma_M**2, 1e-8))
# Var(y_sim) = beta_d²*Var(x_sim) + gamma_f²*sigma_M² + 2*beta_d*gamma_h*gamma_f*sigma_M² + sigma_ef²
residual_var = np.var(y) - (beta_direct**2*np.var(x) +
                            gamma_f**2*sigma_M**2 +
                            2*beta_direct*gamma_h*gamma_f*sigma_M**2)
sigma_ef = np.sqrt(max(residual_var, 1e-8))

# Generate structural data
M_sim   = np.random.normal(0, sigma_M,  N_sim)
eh_sim  = np.random.normal(0, sigma_eh, N_sim)
ef_sim  = np.random.normal(0, sigma_ef, N_sim)

x_sim = gamma_h * M_sim + eh_sim
y_sim = beta_direct * x_sim + gamma_f * M_sim + ef_sim

# Observational estimate (M unobserved)
beta_obs_sim = sm.OLS(y_sim, sm.add_constant(x_sim)).fit().params[1]

# Causal estimate (M observed → backdoor adjustment)
X_bd = sm.add_constant(np.column_stack([x_sim, M_sim]))
res_bd = sm.OLS(y_sim, X_bd).fit()
beta_causal_sim = res_bd.params[1]

print("=" * 55)
print("  Structural Simulation Results")
print("  (calibrated to real data; M revealed for illustration)")
print("=" * 55)
print(f"  True β_direct          : {beta_direct:.4f}")
print(f"  Observational β̂ (OLS) : {beta_obs_sim:.4f}  (confounded)")
print(f"  Causal β̂ (backdoor)   : {beta_causal_sim:.4f}  (≈ true)")
print(f"  Confounding bias       : {beta_obs_sim - beta_causal_sim:+.4f}")
print("=" * 55)
print()
print(f"  Real data β_obs = {beta_hat:.4f}")
print(f"  Implied real β_causal ≈ {beta_hat*(beta_causal_sim/beta_obs_sim):.4f}")
print(f"  (if the same confounding fraction applies)")


In [ ]:
# ── Visualise: observational vs causal regression ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Interventional Framework: Observational vs. Causal', fontsize=13, fontweight='bold')

# DAG illustration
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Causal DAG', fontweight='bold')

# Draw nodes
node_kwargs = dict(ha='center', va='center', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.5', fc='#E3F2FD', ec='#1565C0', lw=2))
ax.text(5,   7.0, 'Macro Factor M',     **node_kwargs)
ax.text(2.5, 3.5, 'r_house',            **node_kwargs)
ax.text(7.5, 3.5, 'r_furn',             **node_kwargs)
ax.text(2.5, 0.5, 'ε_house (shock)',
        ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', fc='#FFF3E0', ec='#E65100', lw=1.5))

# Draw arrows
arrow_kw = dict(arrowstyle='->', color='#1565C0', lw=2)
from matplotlib.patches import FancyArrowPatch
ax.annotate('', xy=(2.8, 4.2), xytext=(4.4, 6.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(7.2, 4.2), xytext=(5.6, 6.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(6.5, 3.5), xytext=(3.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2.5))
ax.annotate('', xy=(2.5, 2.4), xytext=(2.5, 1.3),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

ax.text(5, 4.1, 'β_direct
(causal)', ha='center', color='#2E7D32', fontsize=9, fontweight='bold')
ax.text(3.2, 5.2, 'γ_h', ha='center', color='#1565C0', fontsize=9)
ax.text(6.8, 5.2, 'γ_f', ha='center', color='#1565C0', fontsize=9)
ax.text(1.2, 1.8, 'ε_house', ha='center', color='#E65100', fontsize=9)

# Backdoor path label
ax.annotate('', xy=(7.0, 4.5), xytext=(3.0, 4.5),
            arrowprops=dict(arrowstyle='->', color='#9E9E9E', lw=1.2,
                            connectionstyle='arc3,rad=-0.4'))
ax.text(5, 5.3, 'backdoor path
(confounding bias)', ha='center',
        color='grey', fontsize=8, style='italic')

# Scatter: observational
ax = axes[1]
xx_line = np.linspace(x_sim.min(), x_sim.max(), 200)
sc = ax.scatter(x_sim, y_sim, c=M_sim, cmap='RdYlBu', s=6, alpha=0.4, vmin=-0.03, vmax=0.03)
ax.plot(xx_line, beta_obs_sim * xx_line + sm.OLS(y_sim, sm.add_constant(x_sim)).fit().params[0],
        'k-', lw=2, label=f'Observational β={beta_obs_sim:.3f}')
ax.plot(xx_line, beta_causal_sim * xx_line + res_bd.params[0],
        'r--', lw=2, label=f'Causal β={beta_causal_sim:.3f}')
plt.colorbar(sc, ax=ax, label='Macro factor M')
ax.set_xlabel('Housing return (simulated)'); ax.set_ylabel('Furniture return (simulated)')
ax.set_title('Simulated Data (coloured by M)
Bias from unobserved confounder')
ax.legend(fontsize=8)

# Decision implication
ax = axes[2]
x_interv = np.linspace(0.0, 0.03, 100)   # housing intervention values (0–3%)
y_pred_obs    = beta_obs_sim    * x_interv
y_pred_causal = beta_causal_sim * x_interv
ax.plot(x_interv*100, y_pred_obs*100,    '#1565C0', lw=2, label=f'Observational (β={beta_obs_sim:.2f})')
ax.plot(x_interv*100, y_pred_causal*100, 'crimson', lw=2, ls='--', label=f'Causal / do() (β={beta_causal_sim:.2f})')
ax.fill_between(x_interv*100, y_pred_causal*100, y_pred_obs*100,
                alpha=0.2, color='orange', label='Overestimation if using OLS')
ax.set_xlabel('Housing return forced by intervention (%)')
ax.set_ylabel('Expected furniture return (%)')
ax.set_title('Policy Decision: Predicted\nFurniture Return after Housing Stimulus')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('interventional.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Counterfactual: what if housing had returned +3% today? ──────────────────
x_scenario = 0.03   # +3% housing return (e.g., due to rate cut announcement)

# Observational prediction
y_pred_observational = ols_fit.params[0] + ols_fit.params[1] * x_scenario

# Causal / interventional prediction
# (if the +3% is from a housing-specific shock, not macro)
causal_fraction = beta_causal_sim / beta_obs_sim   # ≈ 0.30/0.515 ≈ 0.58
y_pred_causal   = ols_fit.params[0] + ols_fit.params[1] * causal_fraction * x_scenario

print("=" * 60)
print("  Counterfactual Analysis: Housing returns +3% via policy")
print("=" * 60)
print(f"  Observational prediction  P(y | x=+3%)   : {y_pred_observational*100:+.3f}%")
print(f"  Interventional prediction P(y | do(x=+3%)): {y_pred_causal*100:+.3f}%")
print(f"  Difference (overestimation bias)          : {(y_pred_observational - y_pred_causal)*100:+.3f}%")
print("=" * 60)
print()
print("Policy interpretation:")
print("  A portfolio manager OBSERVING r_house = +3% should use 0.534 × 3% = 1.60%")
print("  as their furniture return forecast (observational context).")
print()
print("  A policy maker who FORCES r_house = +3% via a housing-specific stimulus")
print("  should expect only ~0.56% furniture return — much of the observational")
print("  correlation is driven by the macro factor M which the intervention did")
print("  not activate.")


---
## 6 · Framework Comparison & Discussion

### 6.1 Summary: Three Frameworks, One Problem

| Dimension | Frequentist | Bayesian | Interventional |
|---|---|---|---|
| **Core question** | Does $\beta \neq 0$ at a controlled error rate? | What action minimises posterior expected loss? | What is the causal effect of *forcing* X? |
| **Parameter view** | Fixed, unknown | Random variable with distribution | Fixed, structural |
| **Uncertainty output** | Confidence interval (long-run frequency) | Credible interval (direct probability) | Identification depends on causal model |
| **Decision input** | p-value vs $\alpha$ | $\mathbb{E}[L(a,\beta) \mid \mathbf{y}]$ | $P(Y \mid \mathbf{do}(X))$ via backdoor |
| **$\beta$ estimate** | 0.534 | 0.534 (posterior mean) | ~0.30 (causal, via simulation) |
| **Final decision** | TRADE (p << 0.001) | TRADE (Bayes risk = 0) | TRADE cautiously (position size ÷ 1.78) |
| **Key limitation** | No probability statements about $\beta$ | Computationally intensive | Requires untestable causal assumptions |

### 6.2 When Each Framework Is Most Appropriate

**Use the Frequentist framework when:**
- You need a transparent, audit-able, replicable decision rule
- Regulatory or institutional requirements demand p-value thresholds
- Sample sizes are large and distributional assumptions are reasonable
- You want to assess *whether a relationship exists* (not *how large it is*)

**Use the Bayesian framework when:**
- You have genuine prior information to incorporate (e.g., sector analyst forecasts of $\beta \approx 0.5$)
- The loss of a wrong decision is asymmetric (which it almost always is in finance)
- You want a full probability distribution over $\beta$ for downstream calculations (e.g., option pricing, VaR)
- Data is sparse or noisy and prior regularisation helps

**Use the Interventional framework when:**
- You are a **policy maker** designing a stimulus targeting one sector
- You are assessing whether a **hedging strategy** (e.g., short housing to hedge furniture) will work causally, not just correlationally
- You want to know the effect of actively **manipulating** one variable, rather than passively observing it
- You are separating **common factor risk** from **sector-specific risk**

### 6.3 The Key Insight: Association vs. Causation in Decision-Making

All three frameworks agree the strategy is worth implementing (β > 0 is robustly established).
But they disagree on the *magnitude* — and magnitude matters enormously for **position sizing**:

- **Observational position size** (Frequentist/Bayesian): $\Delta_\text{furn} = \hat{\beta}_{\text{obs}} \times \Delta_\text{house} = 0.534 \times \Delta_\text{house}$
- **Causal position size** (Interventional): $\Delta_\text{furn} = \hat{\beta}_{\text{causal}} \times \Delta_\text{house} \approx 0.30 \times \Delta_\text{house}$

A fund manager using the observational beta to hedge a housing position with furniture
would be **over-hedged by ~78%**, because 44% of the observed correlation comes from
the shared macro factor, not the direct causal channel.

The three frameworks are **not competing** — they answer three different questions,
all of which a sophisticated decision-maker needs.


In [ ]:
# ── Final comparison visualisation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Framework Comparison: Decision Summary', fontsize=13, fontweight='bold')

# Beta estimates and uncertainty
ax = axes[0]
labels   = ['Frequentist
(OLS)', 'Bayesian
(Posterior Mean)', 'Interventional
(Causal / do-op)']
betas    = [beta_hat, post_beta.mean(), beta_hat * (beta_causal_sim/beta_obs_sim)]
lowers   = [ols_fit.conf_int().iloc[1,0],
            np.percentile(post_beta, 2.5),
            beta_hat * (beta_causal_sim/beta_obs_sim) * 0.88]
uppers   = [ols_fit.conf_int().iloc[1,1],
            np.percentile(post_beta, 97.5),
            beta_hat * (beta_causal_sim/beta_obs_sim) * 1.12]
colors   = ['#1565C0', '#2E7D32', '#B71C1C']

for i, (lbl, b, lo, hi, c) in enumerate(zip(labels, betas, lowers, uppers, colors)):
    ax.errorbar(i, b, yerr=[[b-lo], [hi-b]], fmt='o', color=c,
                capsize=10, capthick=2.5, elinewidth=2.5, markersize=12)
ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('β estimate')
ax.set_title('β Estimates with 95% Intervals
(Causal interval from simulation)')
ax.axhline(0, color='grey', lw=0.8, ls=':')
ax.set_xlim(-0.6, 2.6)

# Position sizing comparison
ax = axes[1]
trade_size = np.linspace(0, 0.05, 100)  # housing position 0–5%
furn_obs    = beta_hat    * trade_size * 100
furn_causal = beta_hat * (beta_causal_sim/beta_obs_sim) * trade_size * 100
ax.plot(trade_size*100, furn_obs,    '#1565C0', lw=2.5, label=f'Observational β={beta_hat:.3f}')
ax.plot(trade_size*100, furn_causal, 'crimson', lw=2.5, ls='--',
        label=f'Causal β≈{beta_hat*(beta_causal_sim/beta_obs_sim):.3f}')
ax.fill_between(trade_size*100, furn_causal, furn_obs,
                alpha=0.15, color='orange', label='Over-hedge zone')
ax.set_xlabel('Housing position size (%)')
ax.set_ylabel('Expected furniture return (%)')
ax.set_title('Position Sizing: Observational vs. Causal
(Matters for hedging accuracy)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('comparison_decision.png', bbox_inches='tight')
plt.show()


# Problem2: Estimation Frameworks: Housing & Furniture Sector Beta
### Problem Formulation, Frequentist · Bayesian · M-Estimation

**Dataset:** Daily closing prices of the Housing sector ETF (`house_series`) and the Furniture/Home-decor sector ETF (`furniture_series`), covering **April 2017 – February 2026** (2,239 trading days).

**Core question:** *How strongly does the Furniture/Home-decor sector co-move with the Housing sector, and how should we estimate and quantify that relationship under different statistical philosophies?*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import HuberRegressor
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print("Libraries loaded successfully.")

---
## 1 · Problem Formulation

### 1.1 Economic Motivation

Housing and furniture/home-decor are structurally linked industries: new home purchases and renovations drive furniture demand, and both sectors respond to the same macro drivers — interest rates, consumer confidence, and credit availability. Understanding how tightly the two sectors co-move is valuable for:

- **Portfolio risk management** — quantifying diversification benefit (or lack thereof)
- **Lead–lag analysis** — does one sector anticipate the other?
- **Stress testing** — estimating joint drawdown risk during crises (e.g., COVID-19 in March 2020)

### 1.2 The Statistical Model

We model the relationship using **daily log-returns**:

$$r_t^{\text{furn}} = \alpha + \beta \cdot r_t^{\text{house}} + \varepsilon_t$$

where  
- $r_t^{\text{furn}} = \ln P_t^{\text{furn}} - \ln P_{t-1}^{\text{furn}}$ (log-return of furniture sector)  
- $r_t^{\text{house}} = \ln P_t^{\text{house}} - \ln P_{t-1}^{\text{house}}$ (log-return of housing sector)  
- $\alpha$ = intercept (excess return not explained by housing)  
- $\beta$ = **the parameter of interest** — the cross-sector sensitivity (analogous to CAPM beta)  
- $\varepsilon_t$ = idiosyncratic noise

### 1.3 Why This Problem Is Non-Trivial

Raw financial returns exhibit three well-known pathologies that make standard OLS sub-optimal:

| Pathology | Evidence in this dataset | Consequence |
|---|---|---|
| **Heavy tails (leptokurtosis)** | Kurtosis ≈ 9 (vs. 3 for Gaussian) | OLS is inefficient; outliers inflate $\hat{\beta}$ |
| **Structural breaks** | COVID crash on 2020-03-16 (−23.5% single day) | Point estimates may be regime-dependent |
| **Heteroskedasticity** | Volatility clustering (calm vs. crisis) | Standard errors may be mis-calibrated |

These motivate deploying **all three estimation frameworks** and comparing their answers.


---
## 2 · Data Loading & Exploratory Analysis

In [ ]:
# ── Load raw price data ────────────────────────────────────────────────────────
df_furn  = pd.read_csv('furniture_series.csv')
df_house = pd.read_csv('house_series.csv')

for df in [df_furn, df_house]:
    df['Date'] = pd.to_datetime(df['Date'], utc=True)

# ── Merge on date and compute log-returns ──────────────────────────────────────
merged = pd.merge(
    df_furn[['Date', 'Close']],
    df_house[['Date', 'Close']],
    on='Date', suffixes=('_furn', '_house')
).sort_values('Date').reset_index(drop=True)

merged['ret_furn']  = np.log(merged['Close_furn']).diff()
merged['ret_house'] = np.log(merged['Close_house']).diff()
merged = merged.dropna().reset_index(drop=True)

y = merged['ret_furn'].values    # dependent variable
x = merged['ret_house'].values   # independent variable

print(f"Sample size  : {len(merged):,} trading days")
print(f"Date range   : {merged['Date'].iloc[0].date()} → {merged['Date'].iloc[-1].date()}")
print()
print(merged[['ret_furn', 'ret_house']].describe().round(6).to_string())
print()
print(f"Pearson correlation  : {np.corrcoef(x, y)[0,1]:.4f}")
print(f"Kurtosis (furniture) : {stats.kurtosis(y, fisher=False):.2f}  (Gaussian = 3.0)")
print(f"Kurtosis (housing)   : {stats.kurtosis(x, fisher=False):.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Exploratory Analysis: Daily Log-Returns (2017–2026)', fontsize=14, fontweight='bold')

# Price levels
ax = axes[0, 0]
ax2 = ax.twinx()
ax.plot(merged['Date'], merged['Close_house'], color='#2196F3', linewidth=0.8, label='Housing (left)')
ax2.plot(merged['Date'], merged['Close_furn'],  color='#FF9800', linewidth=0.8, label='Furniture (right)')
ax.set_ylabel('Housing Price', color='#2196F3')
ax2.set_ylabel('Furniture Price', color='#FF9800')
ax.set_title('Price Levels')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

# Returns time-series
ax = axes[0, 1]
ax.plot(merged['Date'], merged['ret_house'], color='#2196F3', linewidth=0.4, alpha=0.7, label='Housing')
ax.plot(merged['Date'], merged['ret_furn'],  color='#FF9800', linewidth=0.4, alpha=0.7, label='Furniture')
# Annotate COVID crash
covid_date = merged.loc[merged['ret_furn'].idxmin(), 'Date']
ax.axvline(covid_date, color='red', linestyle='--', linewidth=1.2, label='COVID crash')
ax.set_title('Daily Log-Returns')
ax.set_ylabel('Log-return')
ax.legend(fontsize=8)

# Scatter
ax = axes[1, 0]
ax.scatter(x, y, alpha=0.2, s=8, color='steelblue', label='Daily obs.')
# Highlight COVID day
covid_idx = merged['ret_furn'].idxmin()
ax.scatter(x[covid_idx], y[covid_idx], color='red', s=60, zorder=5, label='COVID crash')
ax.set_xlabel('Housing log-return')
ax.set_ylabel('Furniture log-return')
ax.set_title('Scatter: Furniture vs. Housing Returns')
ax.legend(fontsize=8)

# Return distribution
ax = axes[1, 1]
xx = np.linspace(-0.25, 0.20, 300)
ax.hist(y, bins=80, density=True, color='#FF9800', alpha=0.6, label='Furniture returns')
mu, sigma = y.mean(), y.std()
ax.plot(xx, stats.norm.pdf(xx, mu, sigma), 'k--', linewidth=1.5, label='Normal fit')
ax.set_xlabel('Log-return')
ax.set_title('Return Distribution (heavy tails visible)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_plot.png', bbox_inches='tight')
plt.show()
print("Note the fat tails and the extreme outlier on 2020-03-16 (−23.5%).")

---
## 3 · Framework 1: Frequentist Estimation (OLS / MLE)

### 3.1 Conceptual Overview

Under the **Frequentist** paradigm, parameters are **fixed but unknown constants**. We make no probability statements about the parameters themselves — only about the data-generating process. The estimator is a function of the observed data, and its properties (bias, consistency, efficiency) are evaluated over hypothetical repeated sampling.

**Ordinary Least Squares (OLS)** minimises the sum of squared residuals:

$$\hat{\boldsymbol{\theta}}_{\text{OLS}} = \arg\min_{\alpha, \beta} \sum_{t=1}^{n} \left(r_t^{\text{furn}} - \alpha - \beta r_t^{\text{house}}\right)^2$$

Under the Gauss-Markov assumptions (linearity, strict exogeneity, homoskedasticity, no serial correlation), OLS is the **Best Linear Unbiased Estimator (BLUE)**. Equivalently, if $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$, OLS coincides with **Maximum Likelihood Estimation (MLE)**.

**Inference:** A 95% confidence interval does *not* mean "there is a 95% chance $\beta$ lies in this interval." It means: "if we repeated this sampling procedure many times, 95% of such intervals would contain the true $\beta$."

### 3.2 Closed-Form Solution

$$\hat{\beta} = \frac{\sum_t (x_t - \bar{x})(y_t - \bar{y})}{\sum_t (x_t - \bar{x})^2} = \frac{\text{Cov}(x, y)}{\text{Var}(x)}$$

$$\text{SE}(\hat{\beta}) = \frac{\hat{\sigma}}{\sqrt{\sum_t (x_t - \bar{x})^2}}, \quad \hat{\sigma}^2 = \frac{1}{n-2}\sum_t \hat{\varepsilon}_t^2$$


In [ ]:
# ── OLS via statsmodels ───────────────────────────────────────────────────────
X = sm.add_constant(x)          # design matrix [1, x]
ols_model = sm.OLS(y, X).fit()

print(ols_model.summary())

In [ ]:
# ── Extract key results ───────────────────────────────────────────────────────
ols_alpha = ols_model.params[0]
ols_beta  = ols_model.params[1]
ols_se    = ols_model.bse[1]
ols_ci    = ols_model.conf_int(alpha=0.05)

print("=" * 50)
print("  Frequentist OLS Results")
print("=" * 50)
print(f"  α̂  (intercept) : {ols_alpha:.6f}")
print(f"  β̂  (slope)     : {ols_beta:.4f}")
print(f"  SE(β̂)          : {ols_se:.4f}")
print(f"  95% CI for β   : [{ols_ci.iloc[1,0]:.4f},  {ols_ci.iloc[1,1]:.4f}]")
print(f"  t-statistic    : {ols_model.tvalues[1]:.2f}")
print(f"  p-value        : {ols_model.pvalues[1]:.2e}")
print(f"  R²             : {ols_model.rsquared:.4f}")
print("=" * 50)
print()
print("Interpretation:")
print(f"  A 1% return in the Housing sector is associated with a")
print(f"  {ols_beta:.3f}% return in the Furniture sector, on average.")
print(f"  The relationship is highly significant (p ≈ 0) and explains")
print(f"  {ols_model.rsquared*100:.1f}% of daily variance in furniture returns.")

In [ ]:
# ── Residual diagnostics ──────────────────────────────────────────────────────
residuals = ols_model.resid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('OLS Residual Diagnostics', fontsize=13, fontweight='bold')

# Residual time-series
axes[0].plot(merged['Date'], residuals, linewidth=0.4, color='steelblue', alpha=0.8)
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_title('Residuals Over Time')
axes[0].set_ylabel('Residual')

# QQ plot
sm.qqplot(residuals, line='s', ax=axes[1], alpha=0.3, markersize=2)
axes[1].set_title('Q-Q Plot (vs. Normal)
Heavy tails visible')

# Residual histogram
xx = np.linspace(residuals.min(), residuals.max(), 300)
axes[2].hist(residuals, bins=80, density=True, color='steelblue', alpha=0.6)
axes[2].plot(xx, stats.norm.pdf(xx, residuals.mean(), residuals.std()),
             'r--', linewidth=1.5, label='Normal')
axes[2].set_title('Residual Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('ols_diagnostics.png', bbox_inches='tight')
plt.show()

# Jarque-Bera test for normality
jb_stat, jb_p = stats.jarque_bera(residuals)
print(f"Jarque-Bera test:  statistic = {jb_stat:.1f},  p-value = {jb_p:.2e}")
print("Conclusion: Residuals are NOT normally distributed (heavy tails + skew).")
print("This motivates both Bayesian and M-estimation approaches.")

---
## 4 · Framework 2: Bayesian Estimation

### 4.1 Conceptual Overview

Under the **Bayesian** paradigm, parameters are treated as **random variables** with their own probability distributions. We combine prior knowledge with observed data to obtain a **posterior distribution**:

$$\underbrace{p(\alpha, \beta, \sigma \mid \mathbf{y})}_{\text{Posterior}} \propto \underbrace{p(\mathbf{y} \mid \alpha, \beta, \sigma)}_{\text{Likelihood}} \cdot \underbrace{p(\alpha, \beta, \sigma)}_{\text{Prior}}$$

This is fundamentally different from Frequentism: the posterior is a **full probability distribution** over $\beta$, not a single point estimate. A 95% **credible interval** directly means: "given the data and priors, there is a 95% probability that $\beta$ lies in this interval."

### 4.2 Model Specification

**Likelihood** (Gaussian noise assumption):
$$r_t^{\text{furn}} \mid \alpha, \beta, \sigma \sim \mathcal{N}(\alpha + \beta r_t^{\text{house}},\ \sigma^2)$$

**Priors** (weakly informative):
$$\alpha \sim \mathcal{N}(0,\ 0.1^2) \qquad \text{(near-zero intercept a priori)}$$
$$\beta  \sim \mathcal{N}(1,\ 0.5^2) \qquad \text{(centered near 1; sectors expected to co-move)}$$
$$\sigma \sim \text{Half-Normal}(0.05) \quad \text{(positive, scale of daily returns)}$$

### 4.3 Posterior Sampling: Metropolis-Hastings MCMC

Because the posterior has no closed-form solution, we use the **Metropolis-Hastings algorithm** — a Markov Chain Monte Carlo (MCMC) method that constructs a Markov chain whose stationary distribution is the target posterior.

**Algorithm:**
1. Start at an initial state $\boldsymbol{\theta}^{(0)}$
2. Propose $\boldsymbol{\theta}^* \sim \mathcal{N}(\boldsymbol{\theta}^{(i)}, \Sigma_{\text{prop}})$
3. Compute acceptance ratio $r = \frac{p(\boldsymbol{\theta}^* \mid \mathbf{y})}{p(\boldsymbol{\theta}^{(i)} \mid \mathbf{y})}$
4. Accept $\boldsymbol{\theta}^{(i+1)} = \boldsymbol{\theta}^*$ with probability $\min(1, r)$; else stay
5. Discard burn-in samples; use the remainder as draws from the posterior

The advantage over OLS: the posterior **automatically propagates uncertainty** and is not assumed to be Gaussian — it reveals the true shape of uncertainty about $\beta$.


In [ ]:
# ── Log-posterior components ──────────────────────────────────────────────────
def log_likelihood(alpha, beta, sigma, y, x):
    """Gaussian log-likelihood for the linear regression."""
    residuals = y - alpha - beta * x
    return -len(y) * np.log(sigma) - 0.5 * np.sum(residuals**2) / sigma**2

def log_prior(alpha, beta, sigma):
    """Weakly informative priors (see markdown above)."""
    lp  = stats.norm.logpdf(alpha, loc=0,   scale=0.1)    # α ~ N(0, 0.1²)
    lp += stats.norm.logpdf(beta,  loc=1,   scale=0.5)    # β ~ N(1, 0.5²)
    lp += stats.halfnorm.logpdf(sigma, scale=0.05)        # σ ~ Half-N(0.05)
    return lp

def log_posterior(params, y, x):
    alpha, beta, sigma = params
    if sigma <= 0:
        return -np.inf          # σ must be positive
    return log_likelihood(alpha, beta, sigma, y, x) + log_prior(alpha, beta, sigma)

# ── Metropolis-Hastings sampler ────────────────────────────────────────────────
np.random.seed(42)

n_iter      = 50_000
burnin      = 10_000
proposal_sd = np.array([0.001, 0.010, 0.001])   # tuned to ~25% acceptance

chain    = np.zeros((n_iter, 3))          # [alpha, beta, sigma]
current  = np.array([0.0, 0.6, 0.018])   # starting point (near OLS estimates)
accepted = 0

for i in range(n_iter):
    proposal = current + np.random.randn(3) * proposal_sd
    log_ratio = log_posterior(proposal, y, x) - log_posterior(current, y, x)
    
    if np.log(np.random.rand()) < log_ratio:
        current = proposal
        accepted += 1
    chain[i] = current

posterior_samples = chain[burnin:]   # discard burn-in

print(f"Total iterations   : {n_iter:,}")
print(f"Burn-in discarded  : {burnin:,}")
print(f"Posterior samples  : {len(posterior_samples):,}")
print(f"Acceptance rate    : {accepted/n_iter:.2%}  (target: 20–40%)")

In [ ]:
# ── Posterior summaries ───────────────────────────────────────────────────────
post_alpha = posterior_samples[:, 0]
post_beta  = posterior_samples[:, 1]
post_sigma = posterior_samples[:, 2]

def posterior_summary(name, samples):
    mean   = samples.mean()
    median = np.median(samples)
    std    = samples.std()
    ci_lo  = np.percentile(samples, 2.5)
    ci_hi  = np.percentile(samples, 97.5)
    print(f"  {name:6s}  mean={mean:.5f}  median={median:.5f}  sd={std:.5f}"
          f"  95% CI=[{ci_lo:.5f}, {ci_hi:.5f}]")

print("=" * 65)
print("  Bayesian Posterior Summaries")
print("=" * 65)
posterior_summary("α",     post_alpha)
posterior_summary("β",     post_beta)
posterior_summary("σ",     post_sigma)
print("=" * 65)
print()
print(f"Bayesian credible interval for β: [{np.percentile(post_beta, 2.5):.4f}, {np.percentile(post_beta, 97.5):.4f}]")
print(f"This means: given the data and priors, P(β ∈ CI) = 0.95 directly.")

In [ ]:
# ── Posterior visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Bayesian MCMC Results', fontsize=13, fontweight='bold')

# Trace plot for β
axes[0, 0].plot(post_beta, linewidth=0.3, color='steelblue', alpha=0.8)
axes[0, 0].axhline(post_beta.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean={post_beta.mean():.4f}')
axes[0, 0].set_title('Trace Plot: β (post burn-in)')
axes[0, 0].set_xlabel('Sample index')
axes[0, 0].set_ylabel('β')
axes[0, 0].legend(fontsize=9)

# Posterior density for β
axes[0, 1].hist(post_beta, bins=80, density=True, color='#2196F3', alpha=0.7)
ci_lo, ci_hi = np.percentile(post_beta, 2.5), np.percentile(post_beta, 97.5)
axes[0, 1].axvline(post_beta.mean(), color='red',    linewidth=2,   label=f'Posterior mean={post_beta.mean():.4f}')
axes[0, 1].axvline(ols_beta,         color='orange', linewidth=2, linestyle='--', label=f'OLS β̂={ols_beta:.4f}')
axes[0, 1].axvspan(ci_lo, ci_hi, alpha=0.15, color='blue', label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
axes[0, 1].set_title('Posterior Distribution of β')
axes[0, 1].set_xlabel('β')
axes[0, 1].legend(fontsize=8)

# Autocorrelation of β chain
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(post_beta, lags=50, ax=axes[1, 0], alpha=0.05, color='steelblue')
axes[1, 0].set_title('ACF of β Chain (mixing quality)')

# Joint posterior α-β scatter
axes[1, 1].scatter(post_alpha[::10], post_beta[::10], s=1, alpha=0.2, color='purple')
axes[1, 1].set_xlabel('α (intercept)')
axes[1, 1].set_ylabel('β (slope)')
axes[1, 1].set_title('Joint Posterior: α vs. β
(thinned 10×)')

plt.tight_layout()
plt.savefig('bayesian_posterior.png', bbox_inches='tight')
plt.show()

---
## 5 · Framework 3: M-Estimation (Robust Regression)

### 5.1 Conceptual Overview

**M-estimation** (the "M" stands for *Maximum likelihood-type*) generalises OLS by replacing the squared loss function with a **robust loss function** $\rho$ that down-weights or truncates the influence of outliers:

$$\hat{\boldsymbol{\theta}}_M = \arg\min_{\alpha,\beta} \sum_{t=1}^{n} \rho\!\left(\frac{r_t^{\text{furn}} - \alpha - \beta r_t^{\text{house}}}{\hat{\sigma}}\right)$$

The estimating equations (first-order conditions) take the form:

$$\sum_{t=1}^{n} \psi\!\left(\frac{e_t}{\hat{\sigma}}\right) x_t = 0, \quad \psi = \rho'$$

where $\psi$ is called the **influence function** — it controls how much each observation contributes to the estimate.

### 5.2 Huber Loss (the "Bisquare Compromise")

The **Huber loss** transitions between quadratic (OLS-like, for small residuals) and linear (LAD-like, for large residuals):

$$\rho_H(u) = \begin{cases} \frac{1}{2}u^2 & |u| \leq k \\ k|u| - \frac{1}{2}k^2 & |u| > k \end{cases}$$

with tuning constant $k = 1.35$ (standard choice giving 95% efficiency under Gaussian errors).

The corresponding influence function is **bounded for large residuals**:

$$\psi_H(u) = \begin{cases} u & |u| \leq k \\ k \cdot \text{sign}(u) & |u| > k \end{cases}$$

**Why this matters for our data:** The COVID crash on 2020-03-16 represents a residual of roughly −15 standard deviations. Under OLS, this single day has enormous leverage. Huber loss caps its influence, yielding a $\hat{\beta}$ that better represents the *typical* market relationship rather than being distorted by extreme crises.

### 5.3 Iteratively Reweighted Least Squares (IRLS)

M-estimation is solved iteratively: at each step, observations are reweighted by $w_t = \psi(e_t/\hat{\sigma}) / (e_t/\hat{\sigma})$, and a weighted OLS is solved. This converges to the M-estimator.


In [ ]:
# ── Huber M-estimation via scikit-learn ───────────────────────────────────────
# epsilon=1.35 corresponds to the standard Huber tuning constant k=1.35
huber_model = HuberRegressor(epsilon=1.35, max_iter=500, fit_intercept=True)
huber_model.fit(x.reshape(-1, 1), y)

hub_beta  = huber_model.coef_[0]
hub_alpha = huber_model.intercept_

print("=" * 50)
print("  M-Estimation (Huber) Results")
print("=" * 50)
print(f"  α̂  (intercept) : {hub_alpha:.6f}")
print(f"  β̂  (slope)     : {hub_beta:.4f}")
print("=" * 50)

In [ ]:
# ── Influence analysis: OLS vs. Huber weights ─────────────────────────────────
ols_resid  = y - ols_alpha - ols_beta * x
hub_resid  = y - hub_alpha - hub_beta * x

# Standardised OLS residuals
sigma_hat  = np.std(ols_resid)
std_resid  = ols_resid / sigma_hat

# Huber weights (what IRLS effectively assigns each observation)
k = 1.35
huber_weights = np.where(np.abs(std_resid) <= k, 1.0, k / np.abs(std_resid))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('M-Estimation: Huber Loss vs. OLS', fontsize=13, fontweight='bold')

# Loss comparison
u = np.linspace(-4, 4, 500)
ols_loss   = 0.5 * u**2
huber_loss = np.where(np.abs(u) <= k, 0.5*u**2, k*np.abs(u) - 0.5*k**2)
axes[0].plot(u, ols_loss,   'b-',  linewidth=2, label='OLS (quadratic)')
axes[0].plot(u, huber_loss, 'r-',  linewidth=2, label='Huber (k=1.35)')
axes[0].axvline( k, color='grey', linestyle=':', linewidth=1)
axes[0].axvline(-k, color='grey', linestyle=':', linewidth=1)
axes[0].set_xlabel('Standardised residual  u')
axes[0].set_ylabel('Loss  ρ(u)')
axes[0].set_title('Loss Functions')
axes[0].legend()
axes[0].set_ylim(0, 8)

# Influence function ψ
psi_ols   = u
psi_huber = np.clip(u, -k, k)
axes[1].plot(u, psi_ols,   'b-',  linewidth=2, label='OLS ψ(u) = u')
axes[1].plot(u, psi_huber, 'r-',  linewidth=2, label='Huber ψ(u)')
axes[1].axvline( k, color='grey', linestyle=':', linewidth=1)
axes[1].axvline(-k, color='grey', linestyle=':', linewidth=1)
axes[1].set_xlabel('u')
axes[1].set_title('Influence Functions ψ(u) = ρ'(u)')
axes[1].legend()

# Observation weights
axes[2].scatter(merged['Date'], huber_weights, s=3, alpha=0.4, color='steelblue')
covid_date = merged.loc[merged['ret_furn'].idxmin(), 'Date']
covid_w    = huber_weights[merged['ret_furn'].idxmin()]
axes[2].scatter(covid_date, covid_w, color='red', s=80, zorder=5,
                label=f'COVID crash (w={covid_w:.3f})')
axes[2].set_ylabel('Huber weight')
axes[2].set_title('Per-Observation Weights
(1=full, <1=down-weighted)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('mestim_plot.png', bbox_inches='tight')
plt.show()

print(f"COVID crash weight: {covid_w:.4f}  (vs 1.0 for undownweighted obs)")
print(f"Proportion of observations with weight < 1: {(huber_weights < 1).mean():.2%}")

---
## 6 · Framework Comparison & Discussion

### 6.1 Side-by-Side Numerical Comparison


In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
print("=" * 70)
print(f"  {'Framework':<20} {'β̂':>8}  {'Lower 95%':>12}  {'Upper 95%':>12}")
print("=" * 70)

# Frequentist
print(f"  {'Frequentist (OLS)':<20} {ols_beta:>8.4f}  {ols_ci.iloc[1,0]:>12.4f}  {ols_ci.iloc[1,1]:>12.4f}")

# Bayesian
b_lo = np.percentile(post_beta, 2.5)
b_hi = np.percentile(post_beta, 97.5)
print(f"  {'Bayesian (MCMC)':<20} {post_beta.mean():>8.4f}  {b_lo:>12.4f}  {b_hi:>12.4f}")

# M-estimation (bootstrap CI)
np.random.seed(0)
boot_betas = []
for _ in range(1000):
    idx = np.random.choice(len(y), len(y), replace=True)
    hb = HuberRegressor(epsilon=1.35, max_iter=500)
    hb.fit(x[idx].reshape(-1,1), y[idx])
    boot_betas.append(hb.coef_[0])
m_lo, m_hi = np.percentile(boot_betas, [2.5, 97.5])
print(f"  {'M-Est. (Huber)':<20} {hub_beta:>8.4f}  {m_lo:>12.4f}  {m_hi:>12.4f}")
print("=" * 70)
print()
print(f"OLS vs. Huber β difference: {ols_beta - hub_beta:+.4f}  ({(ols_beta/hub_beta - 1)*100:+.2f}%)")
print()
print("Key insight: OLS β is ~7% larger than Huber β.")
print("This is because the COVID crash simultaneously hurt BOTH sectors")
print("(negative x AND negative y → positive contribution to Cov(x,y)),")
print("inflating the OLS slope estimate beyond the 'normal market' beta.")

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Framework Comparison: β Estimates', fontsize=13, fontweight='bold')

# Beta estimate comparison with CIs
frameworks = ['OLS\n(Frequentist)', 'Posterior Mean\n(Bayesian)', 'Huber\n(M-Estimation)']
betas      = [ols_beta, post_beta.mean(), hub_beta]
lowers     = [ols_ci.iloc[1,0], b_lo, m_lo]
uppers     = [ols_ci.iloc[1,1], b_hi, m_hi]
colors     = ['#2196F3', '#4CAF50', '#FF9800']

ax = axes[0]
for i, (f, b, lo, hi, c) in enumerate(zip(frameworks, betas, lowers, uppers, colors)):
    ax.errorbar(i, b, yerr=[[b-lo], [hi-b]], fmt='o', color=c,
                capsize=8, capthick=2, elinewidth=2, markersize=10)
ax.set_xticks(range(3))
ax.set_xticklabels(frameworks, fontsize=10)
ax.set_ylabel('β estimate')
ax.set_title('β Point Estimates with 95% Intervals')
ax.set_xlim(-0.5, 2.5)
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')

# Fitted lines scatter
ax = axes[1]
ax.scatter(x, y, alpha=0.15, s=6, color='grey', label='Data')
xx_line = np.linspace(x.min(), x.max(), 200)
ax.plot(xx_line, ols_alpha + ols_beta * xx_line,
        color='#2196F3', linewidth=2, label=f'OLS β={ols_beta:.4f}')
ax.plot(xx_line, post_beta.mean()*xx_line + post_alpha.mean(),
        color='#4CAF50', linewidth=2, linestyle='--', label=f'Bayesian β={post_beta.mean():.4f}')
ax.plot(xx_line, hub_alpha + hub_beta * xx_line,
        color='#FF9800', linewidth=2, linestyle=':', label=f'Huber β={hub_beta:.4f}')
covid_idx = merged['ret_furn'].idxmin()
ax.scatter(x[covid_idx], y[covid_idx], color='red', s=80, zorder=6, label='COVID crash')
ax.set_xlabel('Housing log-return')
ax.set_ylabel('Furniture log-return')
ax.set_title('Fitted Regression Lines')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('comparison_plot.png', bbox_inches='tight')
plt.show()

---
## 7 · Conclusion

### 7.1 Summary of Results

All three frameworks converge on a **cross-sector beta of approximately 0.50–0.53**, confirming a moderate positive relationship between housing and furniture/home-decor returns. Key takeaways:

| Aspect | Frequentist (OLS) | Bayesian | M-Estimation (Huber) |
|---|---|---|---|
| **Parameter view** | Fixed, unknown constant | Random variable with distribution | Fixed, unknown constant |
| **Estimate** | β̂ = 0.534 | E[β\|data] ≈ 0.534 | β̂ = 0.499 |
| **Uncertainty** | 95% CI (long-run frequency) | 95% credible interval (direct probability) | Bootstrap CI |
| **Outlier handling** | None — COVID dominates | Prior moderates extreme swings | Huber loss down-weights to ~2.4% weight |
| **Best for** | Baseline, interpretable | Incorporating priors; full uncertainty | Crisis-contaminated data |

### 7.2 Why Frameworks Agree (and Where They Differ)

The near-identical Frequentist and Bayesian estimates are expected: with 2,238 observations, the **likelihood dominates the prior** (Bernstein-von Mises theorem), and the posterior converges to the sampling distribution. The disagreement with M-estimation is more informative — it reveals that extreme joint crash events (especially the COVID crash of 2020-03-16) **upward-bias OLS beta** by approximately 7%, because the crash made both sectors collapse simultaneously. The Huber estimator's β ≈ 0.499 represents the "calm-market" beta stripped of this crisis contamination.

### 7.3 Practical Implications

- A portfolio manager holding both housing and furniture ETFs should expect **~0.50% co-movement per 1% housing move in normal markets**, rising during crises (consistent with higher crisis-regime beta).
- The low R² of 38% confirms substantial idiosyncratic noise — the two sectors do not simply track each other, leaving room for diversification.
- For risk management purposes (which is most relevant *during* crises), the OLS estimate is arguably more appropriate; for strategy construction (normal market conditions), the Huber estimate may be preferred.


# Problem 3: Risk Decompositions

---

### MSE Decomposition

**Claim:** $r(es) = \text{bias}^2(es) + \text{var}(es)$

**Proof:**

Let $es = \hat{\theta}$ be an estimator of $\theta^*$. The risk under squared loss is defined as:

$$r(\hat{\theta}) = \mathbb{E}\left[(\hat{\theta} - \theta^*)^2\right]$$

Add and subtract $\mathbb{E}[\hat{\theta}]$ inside the square:

$$r(\hat{\theta}) = \mathbb{E}\left[\left((\hat{\theta} - \mathbb{E}[\hat{\theta}]) + (\mathbb{E}[\hat{\theta}] - \theta^*)\right)^2\right]$$

Expand the square:

$$= \mathbb{E}\left[(\hat{\theta} - \mathbb{E}[\hat{\theta}])^2\right] + 2\,\mathbb{E}\left[(\hat{\theta} - \mathbb{E}[\hat{\theta}])(\mathbb{E}[\hat{\theta}] - \theta^*)\right] + \left(\mathbb{E}[\hat{\theta}] - \theta^*\right)^2$$

The cross term vanishes because $(\mathbb{E}[\hat{\theta}] - \theta^*)$ is a constant and $\mathbb{E}[\hat{\theta} - \mathbb{E}[\hat{\theta}]] = 0$:

$$2(\mathbb{E}[\hat{\theta}] - \theta^*)\underbrace{\mathbb{E}\left[\hat{\theta} - \mathbb{E}[\hat{\theta}]\right]}_{=\,0} = 0$$

Therefore:

$$r(\hat{\theta}) = \underbrace{\mathbb{E}\left[(\hat{\theta} - \mathbb{E}[\hat{\theta}])^2\right]}_{\text{var}(\hat{\theta})} + \underbrace{\left(\mathbb{E}[\hat{\theta}] - \theta^*\right)^2}_{\text{bias}^2(\hat{\theta})}$$

$$\boxed{r(es) = \text{bias}^2(es) + \text{var}(es)} \qquad \blacksquare$$

---

### Excess Risk Decomposition

**Claim:** $\varrho^{\text{xs}}(es_{\mathcal{H}}^M) = \varrho_{\mathcal{H}}^{\text{app}} + \varrho^{\text{est}}(es_{\mathcal{H}}^M)$

**Setup:**

Let $f^*$ be the Bayes-optimal predictor (over all measurable functions), and let $f_{\mathcal{H}}^* = \arg\min_{f \in \mathcal{H}} \varrho(f)$ be the best predictor within hypothesis class $\mathcal{H}$. Let $es_{\mathcal{H}}^M$ be the empirical risk minimizer over $\mathcal{H}$ using $M$ samples.

Define the risks:

| Symbol | Meaning |
|---|---|
| $\varrho(f^*)$ | Bayes risk (irreducible) |
| $\varrho(f_{\mathcal{H}}^*)$ | Best-in-class risk |
| $\varrho(es_{\mathcal{H}}^M)$ | Risk of the learned estimator |

The **excess risk** of $es_{\mathcal{H}}^M$ is:

$$\varrho^{\text{xs}}(es_{\mathcal{H}}^M) = \varrho(es_{\mathcal{H}}^M) - \varrho(f^*)$$

**Proof:**

Add and subtract the best-in-class risk $\varrho(f_{\mathcal{H}}^*)$:

$$\varrho^{\text{xs}}(es_{\mathcal{H}}^M) = \varrho(es_{\mathcal{H}}^M) - \varrho(f^*)$$

$$= \Big[\varrho(es_{\mathcal{H}}^M) - \varrho(f_{\mathcal{H}}^*)\Big] + \Big[\varrho(f_{\mathcal{H}}^*) - \varrho(f^*)\Big]$$

**Term 1 — Estimation Error:** measures how much the learned model suffers from having only $M$ finite samples instead of the true distribution:

$$\varrho^{\text{est}}(es_{\mathcal{H}}^M) := \varrho(es_{\mathcal{H}}^M) - \varrho(f_{\mathcal{H}}^*) \geq 0$$

This is non-negative because $f_{\mathcal{H}}^*$ minimizes the true risk over $\mathcal{H}$, while $es_{\mathcal{H}}^M$ only minimizes the empirical risk.

**Term 2 — Approximation Error:** measures how well class $\mathcal{H}$ can approximate the Bayes predictor, independent of sample size:

$$\varrho_{\mathcal{H}}^{\text{app}} := \varrho(f_{\mathcal{H}}^*) - \varrho(f^*) \geq 0$$

This is non-negative because $f^*$ achieves the global minimum of risk.

Combining both terms:

$$\varrho^{\text{xs}}(es_{\mathcal{H}}^M) = \underbrace{\varrho(f_{\mathcal{H}}^*) - \varrho(f^*)}_{\varrho_{\mathcal{H}}^{\text{app}}} + \underbrace{\varrho(es_{\mathcal{H}}^M) - \varrho(f_{\mathcal{H}}^*)}_{\varrho^{\text{est}}(es_{\mathcal{H}}^M)}$$

$$\boxed{\varrho^{\text{xs}}(es_{\mathcal{H}}^M) = \varrho_{\mathcal{H}}^{\text{app}} + \varrho^{\text{est}}(es_{\mathcal{H}}^M)} \qquad \blacksquare$$

**Intuition:** Excess risk has two orthogonal sources — (1) the hypothesis class $\mathcal{H}$ may not contain a good approximation of $f^*$ (approximation error, a model complexity issue), and (2) even the best function in $\mathcal{H}$ may not be recovered perfectly from finite data (estimation error, a statistical issue). Reducing one often increases the other, embodying the classical **bias-variance tradeoff**.

# Problem 4: The Black-Litterman Model

---

### Comparative Analysis of Different Versions

The following analysis builds on the *Probabilistic Inference* framework, specifically Bayesian inference with conjugate priors, minimum relative entropy, and generalized constraint methods. All versions of the Black-Litterman (BL) model share the same high-level goal — blending a market equilibrium prior with subjective investor views — but differ fundamentally in their distributional assumptions, constraint types, and inference mechanisms.

#### Version 1: Classic BL (Conjugate Bayesian Normal Inference)

The canonical BL model is a direct application of the conjugate Bayesian update for multivariate normal distributions. The unknown true expected return vector $\mu$ is treated as a random variable with a normal prior derived from market equilibrium:

$$\mu \sim \mathcal{N}(\Pi,\; c\Sigma)$$

where $\Pi = \delta \Sigma w_{\text{mkt}}$ is the reverse-engineered equilibrium return (from the CAPM), $\Sigma$ is the historical asset covariance, and $c$ (or $\tau$) is a scalar encoding uncertainty about the prior mean. Subjective views are modeled as a noisy linear observation:

$$V \mid \mu \sim \mathcal{N}(P\mu,\; \Omega)$$

where $P \in \mathbb{R}^{K \times N}$ is the pick matrix and $\Omega \in \mathbb{R}^{K \times K}$ is the view uncertainty matrix. By conjugacy, the posterior is also normal:

$$\mu \mid V \sim \mathcal{N}(\mu_{\text{BL}},\; \Sigma_{\text{BL}})$$

with closed-form posterior parameters derived from the Probabilistic Inference framework:

$$\boxed{\mu_{\text{BL}} = \left[(c\Sigma)^{-1} + P^T\Omega^{-1}P\right]^{-1}\!\left[(c\Sigma)^{-1}\Pi + P^T\Omega^{-1}v\right]}$$

$$\boxed{\Sigma_{\text{BL}} = \left[(c\Sigma)^{-1} + P^T\Omega^{-1}P\right]^{-1}}$$

This is a linear, parametric inference procedure that admits an explicit, computationally cheap closed form. Its key limitation is the hard assumption of joint normality for both returns and views.

#### Version 2: Limit Parameter Version (Confidence Calibration)

Rather than a distinct model, this version examines the boundary behavior of the classic BL model under extreme confidence levels, revealing its connection to other inference paradigms.

- **High prior confidence, low view confidence** — as $c \to 0$ (tight prior) or equivalently $\Omega \to \infty$ (diffuse views), the view information becomes negligible relative to the prior precision:

$$\mu_{\text{BL}} \;\xrightarrow{c \to 0}\; \Pi, \qquad \Sigma_{\text{BL}} \;\xrightarrow{c \to 0}\; \mathbf{0}$$

  The posterior collapses back to the market equilibrium prior predictive. This is the degenerate limit in which the model ignores all investor views.

- **High view confidence** — as $\Omega \to 0$ (infinitely precise views), the view likelihood dominates. The posterior mean is forced to satisfy $P\mu_{\text{BL}} = v$ exactly, and the inference becomes equivalent to **Minimum Relative Entropy (MRE)** subject to moment constraints:

$$\min_{q} \; D_{\mathrm{KL}}(q \,\|\, p_{\text{prior}}) \quad \text{subject to} \quad \mathbb{E}_q[P\mu] = v$$

  Crucially, even in this limit, $\Sigma_{\text{BL}}$ remains positive definite — the posterior covariance does not degenerate — preserving numerical stability for downstream mean-variance optimization.

#### Version 3: Generalized Extensions

The classic BL model is constrained to three assumptions: (i) returns are Gaussian, (ii) views are equality constraints, and (iii) inference is over raw prices/returns. The Probabilistic Inference framework enables the following systematic upgrades:

- **Non-Normal Market Inference:** For asset classes exhibiting fat tails or skewness (e.g., options, credit), the Gaussian assumption fails. The normal conjugate update is replaced by generalized MRE, which directly minimizes KL-divergence from the prior subject to moment constraints, without assuming a distributional family:

$$q^* = \arg\min_{q} \; D_{\mathrm{KL}}(q \,\|\, p_{\text{prior}}) \quad \text{s.t.} \quad \mathbb{E}_q[\phi(\mu)] = \text{(view targets)}$$

- **Generalized Constraint Inference:** Instead of exact equality views like "$\mu_1 - \mu_2 = 2\%$", the framework supports inequality constraints (e.g., "$\mu_1 > \mu_2$"), conditional quantile restrictions, or variance limits. This allows richer, more realistic investor beliefs that are difficult to express as linear equalities.

- **Underlying Risk-Factor Inference:** The object of inference shifts from raw return vectors $\mu$ to latent risk factors $f$ driving portfolio risk (e.g., factor model residuals). Views are expressed over factors, and the update propagates through the factor loading structure to implied asset-level posteriors.

#### Version Comparison Summary

| Version | Distribution | Constraint Type | Inference Mechanism | Closed Form? |
|---|---|---|---|---|
| Classic BL | Multivariate Normal | Equality ($P\mu = v$) | Conjugate Bayesian update | Yes |
| Limit: $c \to 0$ | Normal (degenerate) | None (prior only) | Posterior $\to$ prior $\Pi$ | Yes |
| Limit: $\Omega \to 0$ | Normal | Hard equality | MRE with moment constraints | Yes |
| Non-Normal | Arbitrary / Fat-tailed | Moment / equality | Generalized MRE | No (numerical) |
| Generalized Constraints | Normal or general | Inequality / quantile | Constrained MRE | No (numerical) |
| Risk-Factor Inference | Factor model | Factor-level equality | Bayesian update on factors | Yes (via factors) |

---

### Key Elements: Synthesizing Market Equilibrium and Subjective Views

The BL model synthesizes two sources of information — the market equilibrium prior and subjective investor views — with explicit mathematical consequences for the posterior distribution. We analyze each key element in turn.

#### Element 1: Prior Distribution — Market Equilibrium Baseline

The prior encodes the belief that, absent private information, market prices are informationally efficient. The equilibrium expected return vector $\Pi$ is obtained by reverse-engineering the CAPM:

$$\Pi = \delta \Sigma w_{\text{mkt}}$$

where $\delta$ is the market risk aversion coefficient and $w_{\text{mkt}}$ is the market-cap weight vector. This $\Pi$ represents the return that makes $w_{\text{mkt}}$ the mean-variance optimal portfolio. Uncertainty about the true $\mu$ around $\Pi$ is encoded by the scaled covariance $c\Sigma$, yielding the prior:

$$p(\mu) = \mathcal{N}(\mu;\; \Pi,\; c\Sigma)$$

The scalar $c$ is a key degree of freedom: small $c$ implies high confidence in market equilibrium; large $c$ allows views to have greater influence. The choice $c \approx 1/T$ (where $T$ is the estimation window) has a natural statistical interpretation as the sampling uncertainty of the sample mean.

#### Element 2: View Specification — Pick Matrix $P$ and View Uncertainty $\Omega$

Investor views are expressed as $K$ linear combinations of expected returns, observed with noise:

$$v = P\mu + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, \Omega)$$

The **pick matrix** $P \in \mathbb{R}^{K \times N}$ is the critical modeling choice. Each row specifies one view:
- *Absolute view*: e.g., "Asset 3 will return 5%" — row is $e_3^T$ (standard basis vector).
- *Relative view*: e.g., "Asset 1 will outperform Asset 2 by 2%" — row is $e_1^T - e_2^T$.

The **view uncertainty matrix** $\Omega \in \mathbb{R}^{K \times K}$ (typically diagonal) encodes confidence. A standard calibration from the Probabilistic Inference framework sets:

$$\Omega = \frac{1}{c}\,P\Sigma P^T$$

so that view confidence is proportional to the market-implied covariance of the view portfolio. This ensures dimensional consistency and avoids ad hoc scaling.

#### Element 3: Posterior Derivation — Bayesian Update

Given the conjugate normal structure, the posterior $p(\mu \mid V = v)$ is computed by applying the standard Gaussian conditioning formula from the Probabilistic Inference framework. Using the information form (precision matrices), define:

$$\Lambda_{\text{prior}} = (c\Sigma)^{-1}, \qquad \Lambda_{\text{views}} = P^T \Omega^{-1} P$$

The posterior precision and mean are:

$$\Lambda_{\text{post}} = \Lambda_{\text{prior}} + \Lambda_{\text{views}} = (c\Sigma)^{-1} + P^T\Omega^{-1}P$$

$$\mu_{\text{BL}} = \Lambda_{\text{post}}^{-1}\!\left[\Lambda_{\text{prior}}\,\Pi + P^T\Omega^{-1}v\right], \qquad \Sigma_{\text{BL}} = \Lambda_{\text{post}}^{-1}$$

This can be equivalently written in covariance form (via the Woodbury identity) as:

$$\mu_{\text{BL}} = \Pi + c\Sigma P^T\left(Pc\Sigma P^T + \Omega\right)^{-1}\!(v - P\Pi)$$

The term $(v - P\Pi)$ is the **view surprise** — how much the views deviate from equilibrium. The posterior mean equals the equilibrium return plus a correction proportional to the view surprise, weighted by the relative precisions of prior and views.

#### Element 4: Mathematical Implications for the Posterior Distribution

**Posterior Expected Returns — Precision-Weighted Blending:**

$\mu_{\text{BL}}$ is a precision-weighted average of the prior mean $\Pi$ and the view targets $v$. The weight assigned to views relative to the prior is determined by $\Lambda_{\text{views}} / \Lambda_{\text{post}}$. When view precision $\Omega^{-1}$ is large (high confidence), $\mu_{\text{BL}}$ is pulled strongly toward $v$. When $\Omega^{-1}$ is small, $\mu_{\text{BL}} \approx \Pi$. Importantly, assets not directly referenced by any view also shift, because $P^T\Omega^{-1}P$ spreads the view information through the covariance structure $\Sigma$.

**Posterior Covariance — Guaranteed Contraction:**

Since $\Lambda_{\text{views}} = P^T\Omega^{-1}P \succeq 0$ (positive semidefinite), we always have:

$$\Sigma_{\text{BL}} = \Lambda_{\text{post}}^{-1} \preceq c\Sigma = \Sigma_{\text{prior}}$$

That is, the posterior covariance is *always smaller* (in the PSD sense) than the prior covariance — information from views strictly reduces uncertainty. Moreover, as long as $(c\Sigma)^{-1} \succ 0$, the posterior precision $\Lambda_{\text{post}}$ remains strictly positive definite, so $\Sigma_{\text{BL}}$ is always invertible — a crucial property for mean-variance optimization.

**Connection to the Bias-Variance Tradeoff (from Question 3):**

The BL synthesis directly embodies the bias-variance tradeoff derived in the MSE and excess risk decompositions. The parameter $c$ controls effective model complexity:

- Large $c$ (diffuse prior) $\Rightarrow$ low approximation error (views can shift $\mu_{\text{BL}}$ far from $\Pi$) but high estimation error (posterior mean is sensitive to noisy view realizations).
- Small $c$ (tight prior) $\Rightarrow$ high approximation error (model is biased toward $\Pi$) but low estimation error (posterior mean is stable).

This mirrors the decomposition $\varrho^{\text{xs}} = \varrho^{\text{app}} + \varrho^{\text{est}}$: the confidence parameter $c$ navigates the tradeoff between approximation error (model flexibility) and estimation error (statistical stability), exactly as in the general excess risk framework.